In [1]:
import pandas as pd
import ssl
import socket
from datetime import datetime
from tqdm import tqdm
from urllib.parse import urlparse
import re

tqdm.pandas()

from google.colab import drive
drive.mount('/content/drive')

# Load the data
df = pd.read_excel("drive/MyDrive/WildcardLeadApr30New.xls")

def get_ssl_info(hostname):
    port = 443
    if not hostname or pd.isna(hostname):
        return ""
    try:
        hostname = str(hostname).strip()
        # Remove http/https if present
        hostname = hostname.replace('http://', '').replace('https://', '')
        # Remove paths and ports
        hostname = hostname.split('/')[0].split(':')[0]

        context = ssl.create_default_context()
        context.timeout = 10

        with socket.create_connection((hostname, port), timeout=10) as sock:
            with context.wrap_socket(sock, server_hostname=hostname) as ssock:
                cert = ssock.getpeercert()
                return cert
    except Exception as e:
        print(f"Error getting SSL info for {hostname}: {str(e)}")
        return ""

def extract_issuer_organization(cert):
    if not cert or not isinstance(cert, dict):
        return ""

    # Try multiple methods to extract organization name
    issuer = cert.get('issuer', {})

    # Method 1: Standard tuple format
    if isinstance(issuer, tuple):
        for field in issuer:
            for (key, value) in field:
                if key.lower() == 'organizationname':
                    return value

    # Method 2: Dictionary format
    elif isinstance(issuer, dict):
        for key, value in issuer.items():
            if 'organization' in key.lower():
                return value

    # Method 3: String parsing fallback
    issuer_str = str(issuer)
    org_match = re.search(r'O\s*=\s*([^,]+)', issuer_str)
    if org_match:
        return org_match.group(1).strip()

    # Method 4: Subject field fallback
    subject = cert.get('subject', {})
    if isinstance(subject, tuple):
        for field in subject:
            for (key, value) in field:
                if key.lower() == 'organizationname':
                    return value

    return ""

def extract_issuer_common_name(cert):
    if not cert or not isinstance(cert, dict):
        return ""

    # Try multiple methods to extract common name
    issuer = cert.get('issuer', {})
    common_name = ""

    # Method 1: Standard tuple format
    if isinstance(issuer, tuple):
        for field in issuer:
            for (key, value) in field:
                if key.lower() == 'commonname':
                    common_name = value
                    break

    # Method 2: Dictionary format
    elif isinstance(issuer, dict):
        for key, value in issuer.items():
            if 'commonname' in key.lower():
                common_name = value
                break

    # Method 3: String parsing fallback
    if not common_name:
        issuer_str = str(issuer)
        cn_match = re.search(r'CN\s*=\s*([^,]+)', issuer_str)
        if cn_match:
            common_name = cn_match.group(1).strip()

    # Extract only the first/main part of the common name
    if common_name:
        # Remove domain-related parts
        common_name = common_name.split('.')[0]
        # Remove any prefix/suffix
        common_name = re.sub(r'^(www|ssl|secure|wildcard|alpha|beta|test|staging)\.?', '', common_name, flags=re.IGNORECASE)
        # Take only the first word if it's a known CA
        known_cas = ['Sectigo', 'DigiCert', 'GeoTrust', 'Go Daddy', 'GlobalSign', 'Let\'s Encrypt',
                    'COMODO', 'cPanel', 'R3', 'Cloudflare', 'Amazon', 'Google', 'Microsoft']
        for ca in known_cas:
            if ca.lower() in common_name.lower():
                return ca
        # Otherwise return the first word
        return common_name.split()[0] if common_name else ""

    return ""

def extract_wildcard_common_name(cert):
    if not cert or not isinstance(cert, dict):
        return ""

    try:
        # Get the subject field which contains Common Name (CN)
        subject = cert.get('subject', ())

        # For tuple format (common in Python SSL certificates)
        if isinstance(subject, tuple):
            for field in subject:
                for (key, value) in field:
                    if key.lower() == 'commonname' and value.startswith('*.'):
                        return value

        # For dictionary format (less common)
        elif isinstance(subject, dict):
            for key, value in subject.items():
                if 'commonname' in key.lower() and value.startswith('*.'):
                    return value

        return ""
    except Exception:
        return ""

def extract_country(cert):
    if not cert or not isinstance(cert, dict):
        return ""

    # Try multiple methods to extract country
    subject = cert.get('subject', {})
    country = ""

    # Method 1: Standard tuple format
    if isinstance(subject, tuple):
        for field in subject:
            for (key, value) in field:
                if key.lower() == 'countryname':
                    country = value
                    break

    # Method 2: Dictionary format
    elif isinstance(subject, dict):
        for key, value in subject.items():
            if 'countryname' in key.lower():
                country = value
                break

    # Method 3: String parsing fallback
    if not country:
        subject_str = str(subject)
        country_match = re.search(r'C\s*=\s*([^,]+)', subject_str)
        if country_match:
            country = country_match.group(1).strip()

    return country

def format_ssl_date(date_str):
    if not date_str:
        return None

    # Clean the date string
    date_str = date_str.replace('GMT', '').strip()

    formats_to_try = [
        "%b %d %H:%M:%S %Y",
        "%Y-%m-%d %H:%M:%S",
        "%d %b %Y %H:%M:%S",
        "%Y%m%d%H%M%SZ",  # ISO format
        "%Y%m%d%H%M%S"    # ISO format without Z
    ]

    for fmt in formats_to_try:
        try:
            date_obj = datetime.strptime(date_str, fmt)
            return date_obj
        except ValueError:
            continue

    return None  # Return None if can't parse

# Initialize lists to store results
results = []

# Get today's date
today_date = datetime.now().date()

# Process each domain
for domain in tqdm(df["Domain Name"]):
    try:
        ssl_info = get_ssl_info(domain)

        if isinstance(ssl_info, dict):
            not_before = format_ssl_date(ssl_info.get("notBefore", ""))
            not_after = format_ssl_date(ssl_info.get("notAfter", ""))
            organization_name = extract_issuer_organization(ssl_info)
            issuer_common_name = extract_issuer_common_name(ssl_info)
            wildcard_cn = extract_wildcard_common_name(ssl_info)
            country = extract_country(ssl_info)

            # Calculate validity periods
            validity_days = None
            validity_years = None
            today_str = today_date.strftime("%d/%m/%Y")

            # Initialize expiration fields
            days_remaining = None

            if not_before and not_after:
                validity_days = (not_after.date() - not_before.date()).days
                if validity_days < 380:
                    validity_years = "1 year"
                else:
                    validity_years = "2 years"

                # Calculate expiration information
                not_after_date = not_after.date()
                days_diff = (not_after_date - today_date).days
                days_remaining = days_diff

            # Format dates for display
            not_before_str = not_before.strftime("%d/%m/%Y") if not_before else ""
            not_after_str = not_after.strftime("%d/%m/%Y") if not_after else ""

        else:
            not_before_str, not_after_str, organization_name = "", "", ""
            issuer_common_name = ""
            validity_days, validity_years = "", ""
            today_str = today_date.strftime("%d/%m/%Y")
            wildcard_cn = ""
            days_remaining = None
            country = ""

        results.append({
            'Domain': domain,
            'SSL_Not_Before': not_before_str,
            'SSL_Not_After': not_after_str,
            'Organization_Name': organization_name,
            'Common_Name': issuer_common_name,  # Updated to show only starting words
            'Validity_days': validity_days,
            'Validity_years': validity_years,
            'Today_date': today_str,
            'Wild_card_Common_Name': wildcard_cn,
            'days_remaining': days_remaining,
            'Country': country  # New field for country
        })

    except Exception as e:
        print(f"Error processing {domain}: {str(e)}")
        results.append({
            'Domain': domain,
            'SSL_Not_Before': "",
            'SSL_Not_After': "",
            'Organization_Name': "",
            'Common_Name': "",  # Updated to show only starting words
            'Validity_days': "",
            'Validity_years': "",
            'Today_date': today_date.strftime("%d/%m/%Y"),
            'Wild_card_Common_Name': "",
            'days_remaining': None,
            'Country': ""
        })

# Create result dataframe
result_df = pd.DataFrame(results)

# Display preview
print("\nPreview of Results:")
display(result_df.head())

# Save results
result_df.to_excel("drive/MyDrive/Lead Sample_result.xlsx", index=False)
print("\nProcessing complete. Results saved.")


Mounted at /content/drive


FileNotFoundError: [Errno 2] No such file or directory: 'drive/MyDrive/WildcardLeadApr30New.xls'

In [ ]:
result_df

,Domain,SSL_Not_Before,SSL_Not_After,Organization_Name,Common_Name,Validity_days,Validity_years,Today_date,Wild_card_Common_Name,days_remaining,Country
0,wolfltc.com,13/02/2026,14/05/2026,Google Trust Services,R3,90,1 year,30/03/2026,,45.0,
1,apcoa.ie,08/02/2026,09/05/2026,Let's Encrypt,E8,90,1 year,30/03/2026,,40.0,
2,politikenbyrum.dk,17/10/2025,17/10/2026,Sectigo Limited,Sectigo,365,1 year,30/03/2026,*.politikenbyrum.dk,201.0,
3,politikenadvertorial.dk,17/10/2025,17/10/2026,Sectigo Limited,Sectigo,365,1 year,30/03/2026,*.politikenadvertorial.dk,201.0,
4,fdlrez.com,23/02/2026,24/05/2026,Let's Encrypt,R13,90,1 year,30/03/2026,,55.0,
...,...,...,...,...,...,...,...,...,...,...,...
417,NaN,,,,,,,30/03/2026,,NaN,
418,NaN,,,,,,,30/03/2026,,NaN,
419,NaN,,,,,,,30/03/2026,,NaN,
420,NaN,,,,,,,30/03/2026,,NaN,


In [ ]:
df["Domain Name"]

,Domain Name
0,wolfltc.com
1,apcoa.ie
2,politikenbyrum.dk
3,politikenadvertorial.dk
4,fdlrez.com
...,...
417,NaN
418,NaN
419,NaN
420,NaN


In [ ]:
result_df.to_excel("Wildcard Apr 30 Result.xlsx", index=False)

In [ ]:
result_df.to_excel("drive/MyDrive/Wildcard Apr 30 Result.xlsx", index=False)